In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:
import pandas as pd
import numpy as  np
import datetime as dt
import pandas as pd
import numpy as np
import datetime as dt

from Python_arq import engines as engs
from sqlalchemy import text
from calendar import day_name
from sqlalchemy import text
from pathlib import Path

eng = engs.get_engine()
text_qry = text(engs.load_query('base_gradu.sql'))
base_dados = pd.read_sql(text_qry, eng)


In [3]:
base_dados.columns

Index(['ies', 'cód_candidato', 'nome', 'email', 'hubspotcontactid', 'cpf',
       'telefone', 'Data hora inscrição', 'polo', 'curso', 'status',
       'Qtd. HSM', 'Ultima Tabulação', 'qtd_call', 'last_tab'],
      dtype='object')

In [8]:
dia = dt.datetime.now().day
mes = dt.datetime.now().month
ano = dt.datetime.now().year

limiteContato = base_dados['qtd_call'].max()

baseRename = {
    'base_discador':{
    'ies':'IES',
    'cód_candidato':'COD_CANDIDATO',
    'nome':'NOME_CANDIDATO',
    'email':'EMAIL_CANDIDATO',
    'hubspotcontactid':'HUBSPOTCONTACTID',
    'cpf':'CPF_INSCRITO',
    'telefone':'CELULAR',
    'Data hora inscrição':'DATA_INSCRICAO',
    'polo':'POLO',
    'curso':'Curso_Escolhido',
    'Tipo Ingresso':'TIPO_INGRESSO',
    'status':'ANO_MAIOR_NOTA',
    'Data do lead':'MAIOR_NOTA',
    'Data da Tabulação':'DESCONTO',
    'last_tab':'FAIXA_DESCONTO'}
}

clusters = {
    'PUCPR':1679,
    'EADUNISINOS':1680,
    'FAESA':1681,
    'UCS':1681,
    'UNISAGRADO':1682,
    'UNIVALI':1682
}



In [6]:
print(limiteContato)

30


In [ ]:
def padronizarBase(base_discador):
    base_discador.columns = (
        base_discador.columns
        .str.strip()
        .str.lower()
    )

    base_discador = base_discador.rename(
        columns=baseRename['base_discador']
    )

    dropCols = ['Qtd. HSM', 'qtd_call', 'Ultima Tabulação']
    base_discador = base_discador.drop(columns=dropCols, errors='ignore')

    return base_discador


def gerarBasesDiscador(base_discador, clusters, limiteContato=None):
    if limiteContato is not None:
        base_discador = base_discador[base_discador['qtd_call'] <= limiteContato]
    base_discador = padronizarBase(base_discador)

    for ies, campaingid in clusters.items():
        baseDadosFiltrada = base_discador[(base_discador['IES'] == ies)]
        bases = {
            'Inscrito':'Inscrito',
            'Avaliado':'Avaliado',
            'Pré-Matriculado':'Pré-Matriculado',
        }
        for status in bases:
            baseStatusFiltrada = baseDadosFiltrada[baseDadosFiltrada['ANO_MAIOR_NOTA'] == status]
            rows = baseStatusFiltrada.shape[0]
            if rows > 10:
                templateName = fr'CAP_OPS_{campaingid}_{ies}_{ano}{mes}{dia}_{rows}_{status}'
                nomeArquivo = fr'C:\bases\olos\{templateName}.csv'
                print(f'Gerando arquivo: {nomeArquivo} - {rows} linhas')
                try:
                    ##baseStatusFiltrada.to_csv(nomeArquivo, index=False,sep=';')
                    ##print(f'Arquivo gerado com sucesso: {nomeArquivo}')
                    print('...')
                except Exception as e:
                    print(f'Erro ao exportar arquivo: {nomeArquivo} - {str(e)}')    
            

def gerarBasesDisparo(base_dados, clusters, limiteContato=None):
    pass


In [ ]:
import math

def gerarBasesDisparo(base_dados, clusters, limiteContato=None,num_partes=None):

    if limiteContato is None:
        limiteContato = base_dados['Qtd. HSM'].max()
        print(f'Limite de contatos ajustado para {limiteContato} devido ao número máximo de HSMs na base de dados.')

    base_dados = base_dados[base_dados['Qtd. HSM'] <= limiteContato]
    base_dados = base_dados[base_dados['ies'].isin(clusters.keys())]

    if num_partes is None: ##Definir o número de partes para dividir a base 
        num_partes = 4 


    for ies, grupo in base_dados.groupby('ies'):
        
        baseDadosFiltrada = grupo[['telefone','nome']].copy()
        baseDadosFiltrada['nome'] = baseDadosFiltrada['nome'].str.split(' ').str[0].str.capitalize()

        rows = baseDadosFiltrada.shape[0]

        if rows >= 10:  ## Evitar gerar arquivos para bases muito pequenas
            nomeBase = fr'CAP_OPS_GERAL_HSM_{dia}{mes}{ano}_{ies}'
            linhasPorParte = math.ceil(rows / num_partes)
            for i in range(num_partes):
                inicio = i * linhasPorParte
                fim = inicio + linhasPorParte
                parte_df = baseDadosFiltrada.iloc[inicio:fim]
                num_linhas = len(parte_df)
                if num_linhas == 0:
                    continue
                else:
                    nomeArquivo = fr'{nomeBase}_Vol{num_linhas}_Pt{i+1}'
                    caminhoArquivo = fr'C:\bases\disparos\graduacao\{nomeArquivo}.csv'
                    ##parte_df.to_csv(caminhoArquivo, index=False,sep=';', encoding='utf-8-sig')
                    print(f'Gerando arquivo: {nomeBase} | Parte {i + 1} | {num_linhas} linhas.')




In [38]:
gerarBasesDisparo(base_dados,clusters,limiteContato=15,num_partes=4)

Gerando arquivo: CAP_OPS_GERAL_HSM_2742026_EADUNISINOS | Parte 1 | 56 linhas.
Gerando arquivo: CAP_OPS_GERAL_HSM_2742026_EADUNISINOS | Parte 2 | 56 linhas.
Gerando arquivo: CAP_OPS_GERAL_HSM_2742026_EADUNISINOS | Parte 3 | 56 linhas.
Gerando arquivo: CAP_OPS_GERAL_HSM_2742026_EADUNISINOS | Parte 4 | 54 linhas.
Gerando arquivo: CAP_OPS_GERAL_HSM_2742026_FAESA | Parte 1 | 12 linhas.
Gerando arquivo: CAP_OPS_GERAL_HSM_2742026_FAESA | Parte 2 | 12 linhas.
Gerando arquivo: CAP_OPS_GERAL_HSM_2742026_FAESA | Parte 3 | 12 linhas.
Gerando arquivo: CAP_OPS_GERAL_HSM_2742026_FAESA | Parte 4 | 10 linhas.
Gerando arquivo: CAP_OPS_GERAL_HSM_2742026_PUCPR | Parte 1 | 145 linhas.
Gerando arquivo: CAP_OPS_GERAL_HSM_2742026_PUCPR | Parte 2 | 145 linhas.
Gerando arquivo: CAP_OPS_GERAL_HSM_2742026_PUCPR | Parte 3 | 145 linhas.
Gerando arquivo: CAP_OPS_GERAL_HSM_2742026_PUCPR | Parte 4 | 145 linhas.
Gerando arquivo: CAP_OPS_GERAL_HSM_2742026_UCS | Parte 1 | 228 linhas.
Gerando arquivo: CAP_OPS_GERAL_HSM_27